# Centaur IME: 中国語の模型を小さく（d 256・4 層）

小さい模型を 40 分学習し、Rime の答えと組み合わせて選ぶ（rr_zh.py）点を、今の大きさ（zh.pt）と比べる。

使い方: L4 GPU を選び、「すべてのセルを実行」。最初のセルで Google ドライブにつなぐ（許可する）。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/CentaurIME/models/2026-11-07'
import os; os.makedirs(OUT, exist_ok=True); print('保存先', OUT)

In [ ]:
!rm -rf repo && git clone -q --depth 1 -b work/lang-measure https://github.com/J-KAMMA/centaur-ime.git repo && git -C repo log -1 --format='%H %s'
!pip -q install pyarrow==21.0.0 pypinyin==0.53.0 "opencc>=1.1,<2"

In [ ]:
!python repo/tools/fetch_source.py tatoeba-cmn data/zh
!python repo/tools/fetch_source.py wiki-zh data/zh
!python repo/tools/fetch_source.py cc100-zh-Hans data/zh --lines 400000

In [ ]:
import re, pyarrow.parquet as pq, glob
t = pq.read_table(glob.glob('data/zh/*.parquet')[0], columns=['text'])
n = 0
with open('data/zh/wiki.txt', 'w', encoding='utf-8') as f:
    for text in t.column('text').to_pylist():
        for s in re.split(r'[。！？' + chr(10) + ']', text):
            if 4 <= len(s) <= 60:
                f.write(s + chr(10)); n += 1
        if n >= 400000: break
print('ウィキの文', n)

In [ ]:
!python repo/tools/models/train_task.py --task zh --tatoeba data/zh/cmn_sentences.tsv.bz2 --wiki-text data/zh/wiki.txt --cc100 data/zh/cc100-zh-Hans.400000.txt --extra 400000 --batch 256 --oversample 5 --amp --d 256 --layers 4 --heads 4 --minutes 40 --test-limit 300 --out "$OUT" --name zh-d256

In [ ]:
!python repo/tools/models/rr_zh.py --model "$OUT/zh-d256.pt" --rime repo/tools/models/data/rime-chunks-luna_pinyin_simp.tsv --out /content/rrzh
!cp /content/rrzh/rr-zh-result.txt "$OUT/rr-zh-d256-result.txt"

In [ ]:
from google.colab import runtime
runtime.unassign()